# PCD · Qwen2.5-VL-7B · full
Kaggle T4 ×2, Internet. Secret HF_token tùy chọn cho checkpoint công khai.

Run All chạy thẳng toàn bộ Left/Right và Front/Behind, gồm các hoán vị đáp án và ảnh flip. Không chạy sanity inference trước.

Một tiến trình chia checkpoint FP16 qua hai GPU. Lưu prompt gốc/camera, extraction, kiểm tra góc nhìn, phản hồi cả hai nhánh, score và ảnh. Lưu thêm prompt mặt trước/mặt sau và quay hướng nhìn 180°. Prompt quay 180° được ghi riêng, không chấm bằng đáp án gốc. Chạy lại cell cuối để tiếp tục checkpoint khi phiên chạy bị ngắt.


In [ ]:
from pathlib import Path
import os, subprocess, sys
WORK=Path('/kaggle/working/pcd_qwen7b')
WORK.mkdir(exist_ok=True)
TOOLS=Path('/kaggle/temp/pcd7b_tools')
TOOLS.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--target',str(TOOLS),'uv==0.9.5'],check=True)
UV=str(TOOLS/'bin/uv')
ENV=dict(os.environ,UV_PYTHON_INSTALL_DIR='/kaggle/temp/pcd7b_python',UV_CACHE_DIR='/kaggle/temp/pcd7b_uv',
         HF_HOME='/kaggle/temp/pcd7b_hf',HF_HUB_DISABLE_XET='1',TOKENIZERS_PARALLELISM='false')
from kaggle_secrets import UserSecretsClient
try:
    ENV['HF_TOKEN']=UserSecretsClient().get_secret('HF_token')
except Exception:
    print('HF_token unavailable; downloading public checkpoint without authentication.')
VENV=Path('/kaggle/temp/pcd7b_venv')
PY=str(VENV/'bin/python')
if not Path(PY).exists():
    subprocess.run([UV,'venv','--clear','--python','3.11','--managed-python',str(VENV)],env=ENV,check=True)
subprocess.run([PY,'-c','import sys; assert sys.version_info[:2]==(3,11)'],env=ENV,check=True)
subprocess.run([UV,'pip','install','--python',PY,'torch==2.5.1','torchvision==0.20.1',
                '--index-url','https://download.pytorch.org/whl/cu124'],env=ENV,check=True)
subprocess.run([UV,'pip','install','--python',PY,'transformers==4.51.3','accelerate==1.6.0',
    'huggingface_hub==0.30.2','tokenizers==0.21.1','safetensors>=0.4.3,<1',
    'pandas>=2.2,<3','Pillow>=10,<12','tqdm>=4.66,<5','openpyxl>=3.1,<4'],env=ENV,check=True)


## Ghi mã nguồn tự chứa

In [ ]:
(WORK/'pcd_reference.py').write_text('import os\nimport json\nimport argparse\nextraction_trace = {}\nimport base64\nimport re\nfrom io import BytesIO\n\nimport torch\nimport torch.nn.functional as F\nimport pandas as pd\nfrom PIL import Image\nfrom tqdm import tqdm\n\nfrom transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Run PCD on 3DSRBench orientation tasks"\n    )\n    parser.add_argument(\n        "--model_size",\n        type=str,\n        required=True,\n        choices=["3B", "7B", "32B"],\n        help="Model size to use (3B, 7B, or 32B)"\n    )\n    parser.add_argument(\n        "--alpha",\n        type=float,\n        default=4.0,\n        help="Alpha parameter for contrastive decoding"\n    )\n    parser.add_argument(\n        "--data_path",\n        type=str,\n        default="../../data/3dsrbench_v1_vlmevalkit_circular.tsv",\n        help="Path to the TSV data file"\n    )\n    parser.add_argument(\n        "--output_root",\n        type=str,\n        default="../../outputs/3dsrbench",\n        help="Root directory for outputs"\n    )\n    return parser.parse_args()\n\n\ndef decode_image(base64_str):\n    image_data = base64.b64decode(base64_str)\n    return Image.open(BytesIO(image_data)).convert("RGB")\n\n\ndef is_valid_option(x):\n    if x is None:\n        return False\n    if isinstance(x, float) and pd.isna(x):\n        return False\n    return str(x).strip() != ""\n\n\ndef build_options(row):\n    options = {}\n    for k in ["A", "B", "C", "D"]:\n        if k in row and is_valid_option(row[k]):\n            options[k] = str(row[k]).strip()\n    return options, list(options.keys())\n\n\ndef options_to_text(options_dict):\n    return "\\n".join([f"{k}. {v}" for k, v in options_dict.items()])\n\n\nLEFT_RIGHT_EXTRACTION_PROMPT = """\\\nExtract the "reference object" and "target object" from a spatial reasoning question.\n- reference object: the object whose position and facing direction defines the viewpoint\n- target object: the object whose left/right position is being asked about\n\nReturn your answer strictly in this JSON format (no extra text):\n{"ref_obj": "<reference object>", "target_obj": "<target object>"}\n\nExamples:\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the chair\'s position facing where it is facing, is the desk on the left or right of me?\nA: {"ref_obj": "chair", "target_obj": "desk"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the green car\'s position facing where it is facing, is the traffic cone on the left or right of me?\nA: {"ref_obj": "green car", "target_obj": "traffic cone"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the person\'s position facing where it is facing, is the bicycle on the left or right of me?\nA: {"ref_obj": "person", "target_obj": "bicycle"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the woman in pink cloth\'s position facing where it is facing, is the man holding a bag on the left or right of me?\nA: {"ref_obj": "woman in pink cloth", "target_obj": "man holding a bag"}\n\nQ: {question}\nA:"""\n\n\nFRONT_BEHIND_EXTRACTION_PROMPT = """\\\nExtract the "reference object" and "target object" from a spatial reasoning question.\n- reference object: the object whose position and facing direction defines the viewpoint\n- target object: the object whose front/behind position is being asked about\nReturn your answer strictly in this JSON format (no extra text):\n{"ref_obj": "<reference object>", "target_obj": "<target object>"}\n\nExamples:\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the table\'s position facing where it is facing, is the bookshelf in front of me or behind me?\nA: {"ref_obj": "table", "target_obj": "bookshelf"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the truck\'s position facing where it is facing, is the gray car in front of me or behind me?\nA: {"ref_obj": "truck", "target_obj": "stop sign"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the man\'s position facing where it is facing, is the backpack in front of me or behind me?\nA: {"ref_obj": "man", "target_obj": "gray car"}\n\nQ: Consider the real-world 3D locations and orientations of the objects. If I stand at the girl in red dress\'s position facing where it is facing, is the boy with glasses in front of me or behind me?\nA: {"ref_obj": "girl in red dress", "target_obj": "boy with glasses"}\n\nQ: {question}\nA:"""\n\n\n@torch.no_grad()\ndef extract_objects_via_llm(question, category, model, processor, device):\n    if category == "orientation_on_the_left":\n        prompt_template = LEFT_RIGHT_EXTRACTION_PROMPT\n    else:\n        prompt_template = FRONT_BEHIND_EXTRACTION_PROMPT\n    \n    prompt = prompt_template.replace("{question}", question.strip())\n\n    messages = [{\n        "role": "user",\n        "content": [{"type": "text", "text": prompt}],\n    }]\n\n    text = processor.apply_chat_template(\n        messages,\n        tokenize=False,\n        add_generation_prompt=True,\n    )\n\n    inputs = processor(\n        text=[text],\n        padding=True,\n        return_tensors="pt"\n    )\n    for k, v in inputs.items():\n        if torch.is_tensor(v):\n            inputs[k] = v.to(device)\n\n    outputs = model.generate(\n        **inputs,\n        max_new_tokens=64,\n        do_sample=False,\n    )\n\n    generated_text = processor.batch_decode(\n        outputs[:, inputs["input_ids"].shape[1]:],\n        skip_special_tokens=True\n    )[0].strip()\n\n    try:\n        extraction_trace[category,question]=(prompt,generated_text)\n        json_match = re.search(r\'\\{[^{}]*\\}\', generated_text)\n        if json_match:\n            parsed = json.loads(json_match.group())\n            ref_obj = str(parsed.get("ref_obj", "object")).strip()\n            target_obj = str(parsed.get("target_obj", "object")).strip()\n            if ref_obj and target_obj:\n                return ref_obj, target_obj\n    except (json.JSONDecodeError, KeyError, TypeError):\n        pass\n\n    return "object", "object"\n\n\ndef build_rewrite_q(ref_obj, target_obj, category):\n    if category == "orientation_on_the_left":\n        return f"Is the {target_obj} on the left or right of the {ref_obj}?"\n    else:\n        return f"is the {target_obj} in front of the {ref_obj} or behind the {ref_obj}?"\n\n\ndef extract_question_body(question):\n    prefix_patterns = [\n        "Consider the real-world 3D locations and orientations of the objects. ",\n        "Consider the real-world 3D locations and orientations of the objects.\\n",\n    ]\n    \n    question_body = question\n    for pattern in prefix_patterns:\n        if question.startswith(pattern):\n            question_body = question[len(pattern):]\n            break\n    \n    return question_body\n\n\ndef build_original_prompt(question, options_text):\n    prefix = "Consider the real-world 3D locations and orientations of the objects."\n    question_body = extract_question_body(question)\n    return f"""Question: {prefix} {question_body}\n\nOptions:\n{options_text}\n\nReturn only the correct option letter.\n"""\n\n\ndef build_camera_prompt(rewrite_q, options_text):\n    prefix = "Consider the real-world 3D locations and orientations of the objects."\n    return f"""Question: {prefix} From the camera view, {rewrite_q}\n\nOptions:\n{options_text}\n\nReturn only the correct option letter.\n"""\n\n\ndef build_viewpoint_check_prompt(ref_obj, category):\n    if category == "orientation_on_the_left":\n        return (\n            f"Consider the real-world 3D locations and orientations of the objects.\\n"\n            f"Question:\\n"\n            f"In image, are you viewing the {ref_obj}\'s back side?\\n"\n            f"Answer only yes or no."\n        )\n    else:\n        return (\n            f"Question: Consider the real-world 3D locations and orientations of the objects."\n            f"In image, are you looking at the {ref_obj} from behind?"\n            f"Answer only yes or no."\n        )\n\n\ndef build_model_inputs(image, prompt, processor, device):\n    messages = [{\n        "role": "user",\n        "content": [\n            {"type": "image", "image": image},\n            {"type": "text", "text": prompt}\n        ],\n    }]\n\n    text = processor.apply_chat_template(\n        messages,\n        tokenize=False,\n        add_generation_prompt=True,\n    )\n\n    inputs = processor(\n        text=[text],\n        images=[image],\n        padding=True,\n        return_tensors="pt"\n    )\n\n    for k, v in inputs.items():\n        if torch.is_tensor(v):\n            inputs[k] = v.to(device)\n\n    return inputs\n\n\n@torch.no_grad()\ndef generate_answer(image, prompt, model, processor, device):\n    inputs = build_model_inputs(image, prompt, processor, device)\n\n    outputs = model.generate(\n        **inputs,\n        max_new_tokens=20,\n        do_sample=False\n    )\n\n    generated_text = processor.batch_decode(\n        outputs[:, inputs["input_ids"].shape[1]:],\n        skip_special_tokens=True\n    )[0].strip()\n\n    return generated_text\n\n\n@torch.no_grad()\ndef check_same_viewpoint(image, ref_obj, category, model, processor, device):\n    prompt = build_viewpoint_check_prompt(ref_obj, category)\n    raw = generate_answer(image, prompt, model, processor, device)\n    \n    if category == "orientation_on_the_left":\n        is_same = raw.lower().startswith("yes")\n    else:\n        is_same = raw.lower().startswith("no")\n    \n    return raw, is_same\n\n\ndef get_candidate_token_ids_for_label(label, tokenizer):\n    variants = [label, " " + label, "\\n" + label]\n\n    ids = set()\n    for v in variants:\n        tok = tokenizer.encode(v, add_special_tokens=False)\n        if len(tok) == 1:\n            ids.add(tok[0])\n\n    if not ids:\n        tok = tokenizer.encode(label, add_special_tokens=False)\n        ids.add(tok[-1])\n\n    return sorted(ids)\n\n\n@torch.no_grad()\ndef score_next_token_logprobs(image, prompt, model, processor, device):\n    inputs = build_model_inputs(image, prompt, processor, device)\n    outputs = model(**inputs)\n    logits = outputs.logits[:, -1, :]\n    logp = F.log_softmax(logits, dim=-1)[0]\n    return logp\n\n\ndef contrastive_pick_option(image, original_prompt, camera_prompt, allowed_labels, \n                            alpha, model, processor, device, option_token_ids):\n    orig_logp = score_next_token_logprobs(image, original_prompt, model, processor, device)\n    cam_logp = score_next_token_logprobs(image, camera_prompt, model, processor, device)\n\n    scores = {}\n    for label in allowed_labels:\n        ids = option_token_ids[label]\n\n        orig_best = max(orig_logp[i].item() for i in ids)\n        cam_best = max(cam_logp[i].item() for i in ids)\n\n        scores[label] = {\n            "orig": orig_best,\n            "cam": cam_best,\n            "contrastive": orig_best - alpha * cam_best\n        }\n\n    pred = max(scores.items(), key=lambda x: x[1]["contrastive"])[0]\n    return pred, scores\n\n\ndef process_category(df_category, category, model, processor, device, alpha, \n                     model_name_safe, output_dir, option_token_ids):\n    results = []\n    logs = []\n    \n    extraction_cache_path = os.path.join(\n        output_dir,\n        f"{model_name_safe}_{category}_extraction_cache.json"\n    )\n    \n    viewpoint_cache_path = os.path.join(\n        output_dir,\n        f"{model_name_safe}_{category}_viewpoint_cache.json"\n    )\n    \n    if os.path.exists(extraction_cache_path):\n        with open(extraction_cache_path, "r") as f:\n            extraction_cache = json.load(f)\n    else:\n        extraction_cache = {}\n    \n    if os.path.exists(viewpoint_cache_path):\n        with open(viewpoint_cache_path, "r") as f:\n            viewpoint_cache = json.load(f)\n    else:\n        viewpoint_cache = {}\n    \n    for idx in tqdm(range(len(df_category)), desc=f"Processing {category}"):\n        row = df_category.iloc[idx]\n        qid = str(row["qid"])\n        question = str(row["question"])\n\n        image = decode_image(row["image"])\n        options_dict, labels = build_options(row)\n        options_text = options_to_text(options_dict)\n\n        if qid in extraction_cache:\n            ref_obj = extraction_cache[qid]["ref_obj"]\n            target_obj = extraction_cache[qid]["target_obj"]\n        else:\n            ref_obj, target_obj = extract_objects_via_llm(\n                question, category, model, processor, device\n            )\n            extraction_cache[qid] = {"ref_obj": ref_obj, "target_obj": target_obj}\n            with open(extraction_cache_path, "w") as f:\n                json.dump(extraction_cache, f, indent=2, ensure_ascii=False)\n\n        rewrite_q = build_rewrite_q(ref_obj, target_obj, category)\n        \n        original_prompt = build_original_prompt(question, options_text)\n        camera_prompt = build_camera_prompt(rewrite_q, options_text)\n\n        if qid in viewpoint_cache:\n            vp_raw = viewpoint_cache[qid]["raw"]\n            is_same_vp = viewpoint_cache[qid]["is_same"]\n        else:\n            vp_raw, is_same_vp = check_same_viewpoint(\n                image, ref_obj, category, model, processor, device\n            )\n            viewpoint_cache[qid] = {"raw": vp_raw, "is_same": is_same_vp}\n            with open(viewpoint_cache_path, "w") as f:\n                json.dump(viewpoint_cache, f, indent=2, ensure_ascii=False)\n\n        if is_same_vp:\n            pred = generate_answer(image, camera_prompt, model, processor, device)\n            pred_letter = pred.strip()[0].upper() if pred.strip() else "A"\n            if pred_letter not in labels:\n                pred_letter = labels[0]\n            pred = pred_letter\n        else:\n            pred, _ = contrastive_pick_option(\n                image=image,\n                original_prompt=original_prompt,\n                camera_prompt=camera_prompt,\n                allowed_labels=labels,\n                alpha=alpha,\n                model=model,\n                processor=processor,\n                device=device,\n                option_token_ids=option_token_ids\n            )\n\n        answer = str(row["answer"]).strip()\n        hit = int(pred == answer)\n\n        results.append([\n            idx,\n            str(row["qid"]),\n            str(row["question"]),\n            options_dict.get("A", ""),\n            options_dict.get("B", ""),\n            options_dict.get("C", ""),\n            options_dict.get("D", ""),\n            pred,\n            str(row["category"]),\n            rewrite_q,\n            pred,\n            answer,\n            hit\n        ])\n\n        log_entry = {\n            "qid": qid,\n            "category": str(row["category"]),\n            "question": question,\n            "prediction": pred,\n            "answer": answer,\n            "hit": hit\n        }\n        \n        logs.append(log_entry)\n    \n    with open(extraction_cache_path, "w") as f:\n        json.dump(extraction_cache, f, indent=2, ensure_ascii=False)\n    \n    with open(viewpoint_cache_path, "w") as f:\n        json.dump(viewpoint_cache, f, indent=2, ensure_ascii=False)\n    \n    return results, logs\n\n\ndef main():\n    args = parse_args()\n    \n    model_size_map = {\n        "3B": "Qwen/Qwen2.5-VL-3B-Instruct",\n        "7B": "Qwen/Qwen2.5-VL-7B-Instruct",\n        "32B": "Qwen/Qwen2.5-VL-32B-Instruct"\n    }\n    \n    MODEL_NAME = model_size_map[args.model_size]\n    \n    alpha_str = str(args.alpha).replace(".", "p")\n    model_name_safe = f"PCD_Qwen2.5-VL-{args.model_size}-Instruct_alpha{alpha_str}"\n    \n    OUTPUT_DIR = os.path.join(args.output_root, model_name_safe)\n    os.makedirs(OUTPUT_DIR, exist_ok=True)\n    \n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    \n    print(f"\\n{\'=\'*60}")\n    print(f"Model: {MODEL_NAME}")\n    print(f"Alpha: {args.alpha}")\n    print(f"Output Directory: {OUTPUT_DIR}")\n    print(f"Device: {device}")\n    print(f"{\'=\'*60}\\n")\n    \n    print("Loading model...")\n    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n        MODEL_NAME,\n        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n        device_map="auto"\n    )\n    model.eval()\n    \n    processor = AutoProcessor.from_pretrained(MODEL_NAME)\n    tokenizer = processor.tokenizer\n    print("Model loaded successfully!\\n")\n    \n    option_token_ids = {\n        label: get_candidate_token_ids_for_label(label, tokenizer)\n        for label in ["A", "B", "C", "D"]\n    }\n    \n    print(f"Loading data from {args.data_path}...")\n    df = pd.read_csv(args.data_path, sep="\\t")\n    \n    categories = ["orientation_on_the_left", "orientation_in_front_of"]\n    df_filtered = df[df["category"].isin(categories)].reset_index(drop=True)\n    \n    print(f"Total samples: {len(df_filtered)}")\n    for cat in categories:\n        count = len(df_filtered[df_filtered["category"] == cat])\n        print(f"  - {cat}: {count}")\n    print()\n    \n    all_results = []\n    all_logs = []\n    \n    for category in categories:\n        print(f"\\nProcessing category: {category}")\n        \n        df_category = df_filtered[df_filtered["category"] == category].reset_index(drop=True)\n        results, logs = process_category(\n            df_category=df_category,\n            category=category,\n            model=model,\n            processor=processor,\n            device=device,\n            alpha=args.alpha,\n            model_name_safe=model_name_safe,\n            output_dir=OUTPUT_DIR,\n            option_token_ids=option_token_ids\n        )\n        \n        all_results.extend(results)\n        all_logs.extend(logs)\n        \n        cat_acc = sum(log["hit"] for log in logs) / len(logs) if logs else 0\n        print(f"Accuracy: {cat_acc:.4f}")\n    \n    excel_path = os.path.join(\n        OUTPUT_DIR,\n        f"{model_name_safe}_3DSRBenchv1_openai_result.xlsx"\n    )\n    \n    json_path = os.path.join(\n        OUTPUT_DIR,\n        f"{model_name_safe}_results.json"\n    )\n    \n    for i, result in enumerate(all_results):\n        result[0] = i\n    \n    pd.DataFrame(all_results, columns=[\n        "index", "qid", "question", "A", "B", "C", "D",\n        "prediction", "category", "col9", "col10", "answer", "hit"\n    ]).to_excel(excel_path, index=False)\n    \n    with open(json_path, "w") as f:\n        json.dump(all_logs, f, indent=2, ensure_ascii=False)\n    \n    total = len(all_logs)\n    overall_acc = sum(log["hit"] for log in all_logs) / total if total > 0 else 0\n    \n    print(f"\\n{\'=\'*60}")\n    print(f"FINAL RESULTS")\n    print(f"{\'=\'*60}")\n    print(f"Overall accuracy  : {overall_acc:.4f}")\n    \n    for category in categories:\n        cat_logs = [log for log in all_logs if log["category"] == category]\n        cat_acc = sum(log["hit"] for log in cat_logs) / len(cat_logs) if cat_logs else 0\n        print(f"  - {category}: {cat_acc:.4f}")\n    \n    print(f"{\'=\'*60}")\n    print(f"Saved Excel  → {excel_path}")\n    print(f"Saved JSON   → {json_path}")\n    print(f"{\'=\'*60}\\n")\n\n\nif __name__ == "__main__":\n    main()',encoding='utf8')
(WORK/'pcd_kaggle_runner.py').write_text('"""Self-contained Kaggle orchestration; ML dependencies load only inside GPU workers."""\n\nimport base64\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\n\n\nCATEGORIES = ("orientation_on_the_left", "orientation_in_front_of")\nLABELS = ("A", "B", "C", "D")\nEXPORT_COLUMNS = [\n    "index", "qid", "question", "A", "B", "C", "D", "prediction",\n    "category", "col9", "col10", "answer", "hit",\n]\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef circular_id(qid):\n    # Match the repo scorer: remove the option-rotation suffix, retain \'-flip\'.\n    return re.sub(r"-\\d$", "", str(qid))\n\n\ndef get_options(row):\n    return {label: str(row.get(label, "")).strip() for label in LABELS\n            if str(row.get(label, "")).strip()}\n\n\ndef read_benchmark(tsv_path):\n    """Read the official circular TSV, including index-based image references."""\n    csv.field_size_limit(2**30)\n    images, rows, seen = {}, [], set()\n    with Path(tsv_path).open(encoding="utf-8-sig", newline="") as stream:\n        reader = csv.DictReader(stream, delimiter="\\t")\n        required = {"index", "qid", "question", "answer", "category", "image", "A", "B"}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(f"TSV is missing columns: {required - set(reader.fieldnames or [])}")\n        for raw in reader:\n            index = raw["index"].strip()\n            if not index or index in seen:\n                raise ValueError(f"Missing/duplicate TSV index: {index!r}")\n            seen.add(index)\n            images[index] = raw["image"].strip()\n            if raw["category"] not in CATEGORIES:\n                continue\n            row = {key: raw.get(key, "").strip() for key in\n                   ("qid", "question", "answer", "category", *LABELS)}\n            row["row_id"] = index\n            row["base_qid"] = circular_id(row["qid"])\n            options = get_options(row)\n            if len(options) < 2 or row["answer"] not in options:\n                raise ValueError(f"Invalid options/answer for {row[\'qid\']}")\n            rows.append(row)\n    if not rows:\n        raise ValueError("No supported allocentric rows. Use the official circular TSV.")\n    if len({row["qid"] for row in rows}) != len(rows):\n        raise ValueError("Duplicate qid in the selected benchmark.")\n    return rows, images\n\n\ndef resolve_image(index, images):\n    seen = set()\n    while True:\n        if index in seen:\n            raise ValueError(f"Cyclic image reference: {index}")\n        seen.add(index)\n        value = images[index]\n        if value in images:\n            index = value\n            continue\n        if not value:\n            raise ValueError(f"Empty image for index {index}")\n        if value.startswith("data:image/"):\n            value = value.split(",", 1)[1]\n        # Reject a short unresolved numeric index rather than base64-decoding it.\n        if value.isdigit() and len(value) < 32:\n            raise ValueError(f"Missing image reference: {value}")\n        try:\n            return base64.b64decode(value, validate=True)\n        except (ValueError, base64.binascii.Error) as error:\n            raise ValueError(f"Invalid embedded image for index {index}") from error\n\n\ndef validate_circular_groups(rows):\n    groups = {}\n    for row in rows:\n        groups.setdefault(row["base_qid"], []).append(row)\n    for qid, group in groups.items():\n        first_options = get_options(group[0])\n        expected = len(first_options)\n        signatures = {tuple(get_options(row).values()) for row in group}\n        answers = {get_options(row)[row["answer"]] for row in group}\n        if (len(group) != expected or len(signatures) != expected or len(answers) != 1\n                or len({row["category"] for row in group}) != 1\n                or any(sorted(get_options(row).values()) != sorted(first_options.values())\n                       for row in group)):\n            raise ValueError(f"Incomplete/inconsistent circular group {qid}: {len(group)} rows, "\n                             f"expected {expected}. Use the circular TSV without row limits.")\n    return groups\n\n\ndef model_signature(model_dir):\n    model_dir = Path(model_dir)\n    weights = sorted(model_dir.glob("*.safetensors"))\n    if not weights:\n        raise ValueError(f"No safetensors weights in {model_dir}")\n    files = sorted({*weights, *model_dir.glob("*.json"), *model_dir.glob("*.txt"),\n                    *model_dir.glob("*.model"), *model_dir.glob("*.jinja")})\n    # Content hashes also protect resume when a mounted local model changes.\n    return {path.name: sha256_file(path) for path in files}\n\n\ndef runtime_versions():\n    versions = {}\n    for package in ("torch", "torchvision", "transformers", "accelerate", "pandas", "Pillow",\n                    "tokenizers", "huggingface_hub", "safetensors"):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = None\n    return versions\n\n\ndef prepare_run(config):\n    root = Path(config["output_dir"])\n    root.mkdir(parents=True, exist_ok=True)\n    image_dir = root / "images"\n    image_dir.mkdir(exist_ok=True)\n    all_rows, images = read_benchmark(config["data_path"])\n    validate_circular_groups(all_rows)\n    print("Allocentric rows (all rotations and flips):", len(all_rows))\n    for category in CATEGORIES:\n        subset = [row for row in all_rows if row["category"] == category]\n        print(f"  {category}: {len(subset)} rows, "\n              f"{len({row[\'base_qid\'] for row in subset})} circular groups")\n    if config["mode"] == "sanity":\n        selected_qid = config.get("sanity_qid")\n        rows = [row for row in all_rows if row["qid"] == selected_qid] if selected_qid else all_rows[:1]\n        if len(rows) != 1:\n            raise ValueError(f"Sanity qid must select exactly one row: {selected_qid!r}")\n    else:\n        rows = all_rows\n    for row in rows:\n        payload = resolve_image(row["row_id"], images)\n        image_hash = hashlib.sha256(payload).hexdigest()\n        image_path = image_dir / f"{image_hash}.img"\n        if not image_path.exists():\n            image_path.write_bytes(payload)\n        row["image_path"] = str(image_path)\n        row["image_sha256"] = image_hash\n    del images\n    groups = sorted({row["base_qid"] for row in rows})\n    owners = {qid: index % config["num_workers"] for index, qid in enumerate(groups)}\n    manifest = {\n        "schema": 1, "mode": config["mode"], "categories": list(CATEGORIES),\n        "alpha": config["alpha"], "max_pixels": config["max_pixels"],\n        "min_pixels": config["min_pixels"], "compute_baseline": config["compute_baseline"],\n        "num_workers": config["num_workers"], "seed": config["seed"],\n        "dtype": "float16", "attention": "sdpa", "transformers": "4.51.3",\n        "runtime_versions": runtime_versions(),\n        "dataset_sha256": sha256_file(config["data_path"]),\n        "model_files": model_signature(config["model_dir"]),\n        "reference_sha256": sha256_file(config["reference_path"]),\n        "runner_sha256": sha256_file(__file__),\n        "rows": [{key: row[key] for key in ("row_id", "qid", "answer", "image_sha256")}\n                 for row in rows],\n    }\n    # Paths and download locations do not change the scientific run identity.\n    manifest["run_fingerprint"] = fingerprint(manifest)\n    old_manifest = root / "run_manifest.json"\n    if config.get("resume_from") and not old_manifest.exists():\n        previous = Path(config["resume_from"])\n        previous_manifest = read_json(previous / "run_manifest.json")\n        if previous_manifest["run_fingerprint"] != manifest["run_fingerprint"]:\n            raise ValueError("Resume manifest differs. Match the model, data, code, and settings.")\n        for rank in range(config["num_workers"]):\n            source = previous / f"predictions_gpu{rank}.jsonl"\n            destination = root / source.name\n            if source.exists():\n                if destination.exists():\n                    if sha256_file(source) == sha256_file(destination):\n                        continue\n                    raise ValueError(f"Refusing to overwrite existing checkpoint: {destination}")\n                shutil.copy2(source, destination)\n    if old_manifest.exists():\n        if read_json(old_manifest)["run_fingerprint"] != manifest["run_fingerprint"]:\n            raise ValueError("Existing run uses different settings. Choose a new OUTPUT_DIR.")\n    elif any(root.glob("predictions_gpu*.jsonl")) and not config.get("resume_from"):\n        raise ValueError("Predictions exist without a manifest; choose a new OUTPUT_DIR.")\n    write_json(old_manifest, manifest)\n    write_json(root / "config.json", config)\n    for rank in range(config["num_workers"]):\n        shard = rows if config["mode"] == "sanity" else [\n            row for row in rows if owners[row["base_qid"]] == rank]\n        write_json(root / f"plan_gpu{rank}.json", shard)\n    print("Run fingerprint:", manifest["run_fingerprint"])\n    return rows\n\n\ndef read_checkpoint(path, repair_tail=False):\n    """Keep completed rows; only an interrupted, unterminated tail may be repaired."""\n    path = Path(path)\n    if not path.exists():\n        return []\n    records, end = [], 0\n    with path.open("rb") as stream:\n        while line := stream.readline():\n            if not line.endswith(b"\\n"):\n                if not repair_tail:\n                    raise ValueError(f"Interrupted final checkpoint line: {path}")\n                with path.open("r+b") as writable:\n                    writable.truncate(end)\n                break\n            try:\n                records.append(json.loads(line))\n            except (json.JSONDecodeError, UnicodeDecodeError) as error:\n                raise ValueError(f"Corrupt checkpoint at byte {end}: {path}") from error\n            end = stream.tell()\n    if len({row["row_id"] for row in records}) != len(records):\n        raise ValueError(f"Duplicate row_id in checkpoint: {path}")\n    return records\n\n\ndef validate_records(rows, records, run_fingerprint):\n    expected = {row["row_id"]: row for row in rows}\n    for record in records:\n        row = expected.get(record["row_id"])\n        if row is None or record.get("run_fingerprint") != run_fingerprint:\n            raise ValueError("Checkpoint row belongs to a different run/shard.")\n        if any(record.get(key) != row[key] for key in ("qid", "answer", "category")):\n            raise ValueError("Checkpoint question/answer/category differs from the plan.")\n        for prefix in ("", "baseline_"):\n            prediction = record.get(prefix + "prediction")\n            if prediction is None and prefix:\n                continue\n            if (prediction not in get_options(row)\n                    or record.get(prefix + "hit") != int(prediction == row["answer"])):\n                raise ValueError(f"Invalid stored prediction/hit for {row[\'qid\']}")\n\n\ndef parse_letter(text, labels):\n    # Deliberately match process_category() in the reference implementation.\n    letter = text.strip()[0].upper() if text.strip() else "A"\n    fallback = letter not in labels\n    return (labels[0] if fallback else letter), fallback or not text.strip()\n\n\ndef run_worker(config_path, rank):\n    import importlib.util\n    import torch\n    import transformers\n    from PIL import Image\n    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\n\n    config = read_json(config_path)\n    root = Path(config["output_dir"])\n    manifest = read_json(root / "run_manifest.json")\n    if (sha256_file(__file__) != manifest["runner_sha256"]\n            or sha256_file(config["reference_path"]) != manifest["reference_sha256"]\n            or runtime_versions() != manifest["runtime_versions"]):\n        raise RuntimeError("Code/runtime changed after preparation; prepare again in a new OUTPUT_DIR.")\n    rows = read_json(root / f"plan_gpu{rank}.json")\n    checkpoint = root / f"predictions_gpu{rank}.jsonl"\n    completed = read_checkpoint(checkpoint, repair_tail=True)\n    validate_records(rows, completed, manifest["run_fingerprint"])\n    done = {record["row_id"] for record in completed}\n    pending = [row for row in rows if row["row_id"] not in done]\n    print(f"GPU worker {rank}: {len(done)}/{len(rows)} cached; {len(pending)} pending", flush=True)\n    if not pending:\n        return\n    if transformers.__version__ != "4.51.3":\n        raise RuntimeError("Install the pinned transformers version and restart the session.")\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("The single 7B worker must see both GPUs.")\n    torch.cuda.set_device(0)\n    torch.set_num_threads(2)\n    torch.manual_seed(config["seed"])\n    torch.cuda.manual_seed_all(config["seed"])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    device = torch.device("cuda:0")\n    print(f"Physical GPU {rank}: {torch.cuda.get_device_name(0)}; torch={torch.__version__}", flush=True)\n    spec = importlib.util.spec_from_file_location("pcd_reference", config["reference_path"])\n    reference = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(reference)\n    processor_kwargs = {key: config[key] for key in ("min_pixels", "max_pixels")\n                        if config[key] is not None}\n    processor = AutoProcessor.from_pretrained(config["model_dir"], local_files_only=True,\n                                             **processor_kwargs)\n    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n        config["model_dir"], torch_dtype=torch.float16, device_map=\'auto\',max_memory={0:\'11GiB\',1:\'11GiB\'},\n        attn_implementation="sdpa", local_files_only=True, low_cpu_mem_usage=True,\n    ).eval()\n    device = model.get_input_embeddings().weight.device\n    write_json(root/\'model_device_map.json\',model.hf_device_map)\n    if any(str(value) in (\'cpu\',\'disk\') for value in model.hf_device_map.values()):\n        raise RuntimeError(\'Model did not fit both GPUs. Inspect model_device_map.json.\')\n    token_ids = {label: reference.get_candidate_token_ids_for_label(label, processor.tokenizer)\n                 for label in LABELS}\n    original_extract=reference.extract_objects_via_llm\n    original_viewpoint=reference.check_same_viewpoint\n    def logged_extract(question,category,*args,**kwargs):\n        path=active_trace[\'folder\']/\'extraction.json\'\n        write_json(path,dict(question=question,category=category,status=\'started\'))\n        result=original_extract(question,category,*args,**kwargs)\n        prompt,raw=reference.extraction_trace[(category,question)]\n        write_json(path,dict(prompt=prompt,raw_output=raw,objects=result,status=\'complete\'))\n        return result\n    def logged_viewpoint(image,ref_obj,category,*args,**kwargs):\n        path=active_trace[\'folder\']/\'viewpoint.json\'\n        prompt=reference.build_viewpoint_check_prompt(ref_obj,category)\n        write_json(path,dict(prompt=prompt,status=\'started\'))\n        raw,same=original_viewpoint(image,ref_obj,category,*args,**kwargs)\n        write_json(path,dict(prompt=prompt,raw_output=raw,is_same_viewpoint=same,status=\'complete\'))\n        return raw,same\n    reference.extract_objects_via_llm=logged_extract\n    reference.check_same_viewpoint=logged_viewpoint\n    original_generate=reference.generate_answer\n    original_contrastive=reference.contrastive_pick_option\n    active_trace={}\n    def logged_generate(image,prompt,*args,**kwargs):\n        path=active_trace[\'folder\']/f"generate_{active_trace[\'index\']:03d}.json"\n        active_trace[\'index\']+=1\n        write_json(path,dict(prompt=prompt,status=\'started\'))\n        raw=original_generate(image,prompt,*args,**kwargs)\n        write_json(path,dict(prompt=prompt,raw_output=raw,status=\'complete\'))\n        return raw\n    def logged_contrastive(image,original_prompt,camera_prompt,*args,**kwargs):\n        path=active_trace[\'folder\']/f"scores_{active_trace[\'index\']:03d}.json"\n        active_trace[\'index\']+=1\n        write_json(path,dict(original_prompt=original_prompt,camera_prompt=camera_prompt,status=\'started\'))\n        prediction,scores=original_contrastive(image,original_prompt,camera_prompt,*args,**kwargs)\n        write_json(path,dict(original_prompt=original_prompt,camera_prompt=camera_prompt,\n                             prediction=prediction,scores=scores,status=\'complete\'))\n        return prediction,scores\n    reference.generate_answer=logged_generate\n    reference.contrastive_pick_option=logged_contrastive\n    # Extraction is text-only; viewpoint cache keys include the actual (possibly flipped) image.\n    extraction_cache, viewpoint_cache = {}, {}\n    for record in completed:\n        reference.extraction_trace[(record[\'category\'],record[\'question\'])] = (\n            record.get(\'extraction_prompt\',\'\'),record.get(\'extraction_raw\',\'\'))\n        extraction_cache[(record["category"], record["question"])] = (\n            record["ref_obj"], record["target_obj"])\n        viewpoint_cache[(record["category"], record["image_sha256"], record["ref_obj"])] = (\n            record["viewpoint_raw"], record["is_same_viewpoint"])\n    with checkpoint.open("a", encoding="utf-8") as sink, torch.inference_mode():\n        for ordinal, row in enumerate(pending, 1):\n            start = time.monotonic()\n            trace=root/\'trace\'/str(row[\'row_id\'])\n            trace.mkdir(parents=True,exist_ok=True)\n            active_trace.update(folder=trace,index=0)\n            try:\n                torch.cuda.reset_peak_memory_stats()\n                with Image.open(row["image_path"]) as original_image:\n                    image = original_image.convert("RGB")\n                category, question = row["category"], row["question"]\n                options = get_options(row)\n                labels = list(options)\n                extraction_key = (category, question)\n                if extraction_key not in extraction_cache:\n                    extraction_cache[extraction_key] = reference.extract_objects_via_llm(\n                        question, category, model, processor, device)\n                ref_obj, target_obj = extraction_cache[extraction_key]\n                rewritten = reference.build_rewrite_q(ref_obj, target_obj, category)\n                options_text = reference.options_to_text(options)\n                original_prompt = reference.build_original_prompt(question, options_text)\n                camera_prompt = reference.build_camera_prompt(rewritten, options_text)\n                viewpoint_key = (category, row["image_sha256"], ref_obj)\n                if viewpoint_key not in viewpoint_cache:\n                    viewpoint_cache[viewpoint_key] = reference.check_same_viewpoint(\n                        image, ref_obj, category, model, processor, device)\n                viewpoint_raw, is_same = viewpoint_cache[viewpoint_key]\n                scores, camera_raw, fallback = None, None, False\n                if is_same:\n                    camera_raw = reference.generate_answer(image, camera_prompt, model, processor, device)\n                    prediction, fallback = parse_letter(camera_raw, labels)\n                else:\n                    prediction, scores = reference.contrastive_pick_option(\n                        image, original_prompt, camera_prompt, labels, config["alpha"],\n                        model, processor, device, token_ids)\n                if scores is None:\n                    _, scores = reference.contrastive_pick_option(\n                        image, original_prompt, camera_prompt, labels, config["alpha"],\n                        model, processor, device, token_ids)\n                if scores is not None and not all(\n                        math.isfinite(value) for candidate in scores.values() for value in candidate.values()):\n                    raise FloatingPointError("Non-finite option scores; inspect model/dtype before resuming.")\n                record = {**row, "run_fingerprint": manifest["run_fingerprint"],\n                          "gpu_rank": rank, "ref_obj": ref_obj, "target_obj": target_obj,\n                          "extraction_fallback": (ref_obj, target_obj) == ("object", "object"),\n                          "rewrite_question": rewritten, "viewpoint_raw": viewpoint_raw,\n                          "is_same_viewpoint": is_same, "branch": "camera_direct" if is_same else "pcd",\n                          "camera_raw": camera_raw, "answer_parse_fallback": fallback,\n                          "prediction": prediction, "hit": int(prediction == row["answer"]),\n                          "scores": scores}\n                if config["compute_baseline"]:\n                    baseline_raw = reference.generate_answer(image, original_prompt, model, processor, device)\n                    baseline, baseline_fallback = parse_letter(baseline_raw, labels)\n                    record.update(baseline_raw=baseline_raw, baseline_prediction=baseline,\n                                  baseline_hit=int(baseline == row["answer"]),\n                                  baseline_parse_fallback=baseline_fallback)\n                if True:\n                    record.update(original_prompt=original_prompt, camera_prompt=camera_prompt,\n                                  viewpoint_prompt=reference.build_viewpoint_check_prompt(ref_obj, category),\n                                  option_token_ids=token_ids, image_size=list(image.size))\n                if camera_raw is None:\n                    camera_raw = reference.generate_answer(image,camera_prompt,model,processor,device)\n                record["diagnostic_camera_prediction"]=parse_letter(camera_raw,labels)[0]\n                record["diagnostic_camera_raw"]=camera_raw\n                record["diagnostic_contrastive_prediction"]=max(labels,key=lambda label:scores[label]["contrastive"])\n                record["extraction_prompt"],record["extraction_raw"]=reference.extraction_trace[extraction_key]\n                # Counterfactual diagnostic: rotate the reference frame 180 degrees.\n                reverse_question=question.replace(\n                    "facing where it is facing", "facing in the direction opposite to where it is facing")\n                if reverse_question == question:\n                    raise ValueError(\'Cannot construct a 180-degree diagnostic for this question.\')\n                reverse_prompt=reference.build_original_prompt(reverse_question,reference.options_to_text(options))\n                reverse_raw=reference.generate_answer(image,reverse_prompt,model,processor,device)\n                record[\'reverse_direction_prompt\']=reverse_prompt\n                record[\'reverse_direction_raw\']=reverse_raw\n                record[\'reverse_direction_prediction\']=parse_letter(reverse_raw,labels)[0]\n                record[\'reverse_direction_answer\']=None\n                record[\'reverse_direction_protocol\']=\'Reference frame rotated 180 degrees; unscored counterfactual.\'\n                opposite_viewpoint=reference.build_viewpoint_check_prompt(ref_obj,category)\n                opposite_viewpoint=opposite_viewpoint.replace("back side","front side").replace("from behind","from the front")\n                record[\'opposite_viewpoint_prompt\']=opposite_viewpoint\n                record[\'opposite_viewpoint_raw\']=reference.generate_answer(image,opposite_viewpoint,model,processor,device)\n                write_json(trace/\'record.json\',record)\n                torch.cuda.synchronize()\n                record["seconds"] = time.monotonic() - start\n                record["peak_vram_gib"] = torch.cuda.max_memory_allocated() / 2**30\n                sink.write(json.dumps(record, ensure_ascii=False) + "\\n")\n                sink.flush()\n                os.fsync(sink.fileno())\n                print(f"[{len(done) + ordinal}/{len(rows)}] {row[\'qid\']} "\n                      f"pred={prediction} answer={row[\'answer\']} branch={record[\'branch\']} "\n                      f"time={record[\'seconds\']:.2f}s vram={record[\'peak_vram_gib\']:.2f}GiB", flush=True)\n            except Exception as error:\n                import traceback\n                write_json(trace/\'error.json\',dict(error=repr(error),traceback=traceback.format_exc(),row_id=row[\'row_id\']))\n                write_json(root / f"error_gpu{rank}.json", {\n                    "row_id": row["row_id"], "qid": row["qid"], "error": repr(error),\n                    "hint": "Fix the error, then rerun. Completed JSONL rows are preserved. "\n                            "If changing model/settings, use a new OUTPUT_DIR.",\n                })\n                raise\n\n\ndef launch_workers(config):\n    root = Path(config["output_dir"])\n    processes, streams = [], []\n    try:\n        for rank in range(config["num_workers"]):\n            log_path = root / f"worker_gpu{rank}.log"\n            stream = log_path.open("w", encoding="utf-8")\n            streams.append(stream)\n            environment = dict(os.environ, CUDA_VISIBLE_DEVICES=os.environ.get("CUDA_VISIBLE_DEVICES","0,1"),\n                               TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2",\n                               PYTHONUNBUFFERED="1")\n            process = subprocess.Popen(\n                [sys.executable, "-u", str(Path(__file__).resolve()), "--worker",\n                 str(root / "config.json"), str(rank)], env=environment,\n                stdout=stream, stderr=subprocess.STDOUT,\n            )\n            processes.append(process)\n        while True:\n            statuses = [process.poll() for process in processes]\n            for rank, status in enumerate(statuses):\n                if status not in (None, 0):\n                    tail = "\\n".join((root / f"worker_gpu{rank}.log").read_text(\n                        encoding="utf-8", errors="replace").splitlines()[-30:])\n                    raise RuntimeError(f"GPU {rank} exited with status {status}:\\n{tail}")\n            for rank in range(config["num_workers"]):\n                log = (root / f"worker_gpu{rank}.log").read_text(encoding="utf-8", errors="replace")\n                print(f"GPU {rank}: " + (log.splitlines()[-1] if log.splitlines() else "loading..."), flush=True)\n            if all(status == 0 for status in statuses):\n                break\n            time.sleep(15)\n    finally:\n        for process in processes:\n            if process.poll() is None:\n                process.terminate()\n        for process in processes:\n            try:\n                process.wait(timeout=15)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        for stream in streams:\n            stream.close()\n\n\ndef compute_metrics(rows, records, include_baseline=True):\n    groups = validate_circular_groups(rows)\n    if len({record["row_id"] for record in records}) != len(records):\n        raise ValueError("Duplicate predictions across shards.")\n    by_id = {record["row_id"]: record for record in records}\n    expected = {row["row_id"] for row in rows}\n    if set(by_id) != expected:\n        raise ValueError(f"Incomplete run: {len(expected - set(by_id))} missing, "\n                         f"{len(set(by_id) - expected)} unexpected rows. Rerun workers to resume.")\n    report = []\n    for method, hit_key in [("pcd", "hit")] + ([("baseline", "baseline_hit")] if include_baseline else []):\n        for category in ("overall", *CATEGORIES):\n            selected = [row for row in rows if category == "overall" or row["category"] == category]\n            selected_groups = [group for group in groups.values()\n                               if category == "overall" or group[0]["category"] == category]\n            correct = sum(all(by_id[row["row_id"]][hit_key] == 1 for row in group)\n                          for group in selected_groups)\n            report.append({"method": method, "category": category,\n                           "rows": len(selected), "circular_groups": len(selected_groups),\n                           "row_accuracy_pct": 100 * sum(by_id[row["row_id"]][hit_key]\n                                                          for row in selected) / len(selected) if selected else None,\n                           "circular_accuracy_pct": 100 * correct / len(selected_groups) if selected_groups else None})\n    return report\n\n\ndef export_results(config):\n    import pandas as pd\n\n    root = Path(config["output_dir"])\n    run_fingerprint = read_json(root / "run_manifest.json")["run_fingerprint"]\n    all_records, rows = [], []\n    for rank in range(config["num_workers"]):\n        plan = read_json(root / f"plan_gpu{rank}.json")\n        records = read_checkpoint(root / f"predictions_gpu{rank}.jsonl")\n        validate_records(plan, records, run_fingerprint)\n        if len(records) != len(plan):\n            raise ValueError(f"GPU {rank} has {len(records)}/{len(plan)} rows; rerun workers.")\n        all_records.extend(records)\n        if config["mode"] != "sanity" or rank == 0:\n            rows.extend(plan)\n    if config["mode"] == "sanity":\n        write_json(root / "sanity_results.json", all_records)\n        return pd.DataFrame([{key: record.get(key) for key in (\n            "gpu_rank", "qid", "branch", "prediction", "answer", "hit", "baseline_prediction",\n            "viewpoint_raw", "extraction_fallback", "seconds", "peak_vram_gib")}\n                             for record in all_records]), all_records\n    metrics = compute_metrics(rows, all_records, config["compute_baseline"])\n    metric_frame = pd.DataFrame(metrics)\n    metric_frame.to_csv(root / "metrics.csv", index=False)\n    write_json(root / "metrics.json", metrics)\n    all_records.sort(key=lambda row: int(row["row_id"]))\n    write_json(root / "predictions.json", all_records)\n    pd.DataFrame(all_records).to_csv(root / "predictions.csv", index=False)\n    alpha = str(config["alpha"]).replace(".", "p")\n    model_name = f"PCD_Qwen2.5-VL-7B-Instruct_alpha{alpha}"\n    export_dir = root / "outputs" / model_name\n    export_dir.mkdir(parents=True, exist_ok=True)\n    excel_rows = []\n    for index, row in enumerate(all_records):\n        excel_rows.append([index, row["qid"], row["question"], *[row[key] for key in LABELS],\n                           row["prediction"], row["category"], row["rewrite_question"],\n                           row["prediction"], row["answer"], row["hit"]])\n    pd.DataFrame(excel_rows, columns=EXPORT_COLUMNS).to_excel(\n        export_dir / f"{model_name}_3DSRBenchv1_openai_result.xlsx", index=False)\n    write_json(export_dir / f"{model_name}_results.json", all_records)\n    write_json(root / "diagnostics.json", {\n        "rows": len(all_records),\n        "extraction_fallback_rows": sum(row["extraction_fallback"] for row in all_records),\n        "answer_parse_fallback_rows": sum(row["answer_parse_fallback"] for row in all_records),\n        "camera_direct_rows": sum(row["is_same_viewpoint"] for row in all_records),\n        "pcd_rows": sum(not row["is_same_viewpoint"] for row in all_records),\n    })\n    return metric_frame, all_records\n\n\nif __name__ == "__main__":\n    if len(sys.argv) != 4 or sys.argv[1] != "--worker":\n        raise SystemExit("Usage: kaggle_runner.py --worker CONFIG_JSON RANK")\n    run_worker(sys.argv[2], int(sys.argv[3]))',encoding='utf8')


## Cấu hình lượt chạy

In [ ]:
from pathlib import Path

MODE = "full"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_REVISION = "main"
DATASET_ID = "ccvl/3DSRBench"
DATASET_REVISION = "5c90e422b68d0484d50a3ff06b3096ea7d288026"
TSV_FILENAME = "3dsrbench_v1_vlmevalkit_circular.tsv"
ALPHA = 4.0
NUM_WORKERS = 1
SEED = 42
COMPUTE_BASELINE = True  # Original allocentric prompt, greedy decoding; extra inference cost.

# None preserves the repository's default AutoProcessor image resolution.
# For a lower-memory experiment, set MAX_PIXELS = 1280 * 28 * 28 (recorded in manifest).
# Changing this setting can change accuracy; use a new OUTPUT_DIR.
MIN_PIXELS = None
MAX_PIXELS = None

# Optional mounted Kaggle inputs. None tries /kaggle/input, then downloads the public files.
# MODEL_DIR must contain config, tokenizer, processor, and ALL safetensors shards.
MODEL_DIR = None  # e.g. "/kaggle/input/qwen25vl7b/transformers/default/1"
TSV_PATH = None   # e.g. "/kaggle/input/3dsrbench/3dsrbench_v1_vlmevalkit_circular.tsv"
SANITY_QID = None  # None chooses the first supported row; exact qid selects another image.

OUTPUT_DIR = Path("/kaggle/working/pcd_qwen7b") / MODE
# Same session: rerun the worker cell. New session: attach the old output as Kaggle input,
# point RESUME_FROM at the directory containing run_manifest.json + predictions_gpu*.jsonl.
RESUME_FROM = None
assert ALPHA >= 0 and NUM_WORKERS == 1
if MIN_PIXELS is not None and MAX_PIXELS is not None:
    assert 0 < MIN_PIXELS <= MAX_PIXELS

## Chạy trong Python 3.11 riêng và xuất trace

In [ ]:
driver='import json\nimport torch\nfrom huggingface_hub import hf_hub_download, snapshot_download, HfApi\nMODEL_REVISION=HfApi().model_info(MODEL_ID,revision=MODEL_REVISION).sha\n\nassert torch.cuda.is_available(), "Chọn Settings → Accelerator → GPU T4 x2."\nassert torch.cuda.device_count() == 2, f"Cần đúng 2 GPU, thấy {torch.cuda.device_count()}."\nfor gpu in range(2):\n    info = torch.cuda.get_device_properties(gpu)\n    print(f"GPU {gpu}: {info.name}, {info.total_memory / 2**30:.1f} GiB")\nprint("One FP16 7B model spans two GPUs, with SDPA.")\n\ninput_root = Path("/kaggle/input")\nif TSV_PATH is None:\n    candidates = sorted(input_root.rglob(TSV_FILENAME)) if input_root.exists() else []\n    if len(candidates) > 1:\n        raise ValueError("Nhiều TSV được tìm thấy; đặt TSV_PATH rõ ràng.")\n    if candidates:\n        TSV_PATH = str(candidates[0])\n    else:\n        TSV_PATH = hf_hub_download(repo_id=DATASET_ID, filename=TSV_FILENAME,\n                                   repo_type="dataset", revision=DATASET_REVISION)\nif MODEL_DIR is None:\n    candidates = []\n    for path in input_root.rglob("config.json") if input_root.exists() else []:\n        try:\n            metadata = json.loads(path.read_text())\n        except (ValueError, OSError):\n            continue\n        if (metadata.get("model_type") == "qwen2_5_vl"\n                and metadata.get("hidden_size") == 3584\n                and metadata.get("num_hidden_layers") == 28\n                and list(path.parent.glob("*.safetensors"))):\n            candidates.append(path.parent)\n    if len(candidates) > 1:\n        raise ValueError("Nhiều bản model được tìm thấy; đặt MODEL_DIR rõ ràng.")\n    if candidates:\n        MODEL_DIR = str(candidates[0])\n    else:\n        MODEL_DIR = snapshot_download(\n            MODEL_ID, revision=MODEL_REVISION,\n            allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],\n            max_workers=2,\n        )\nmodel_path = Path(MODEL_DIR).resolve()\nmodel_metadata = json.loads((model_path / "config.json").read_text())\nassert (model_metadata.get("model_type"), model_metadata.get("hidden_size"),\n        model_metadata.get("num_hidden_layers")) == ("qwen2_5_vl", 3584, 28), "Expected official Qwen2.5-VL-7B."\nassert (model_path / "preprocessor_config.json").is_file(), "Thiếu processor."\nassert (model_path / "tokenizer_config.json").is_file(), "Thiếu tokenizer."\nindex_path = model_path / "model.safetensors.index.json"\nif index_path.exists():\n    weight_names = set(json.loads(index_path.read_text())["weight_map"].values())\n    assert all((model_path / name).is_file() for name in weight_names), "Thiếu weight shard."\nelse:\n    assert (model_path / "model.safetensors").is_file(), "Thiếu model.safetensors hoặc shard index."\nassert Path(TSV_PATH).is_file(), f"Không tìm thấy TSV: {TSV_PATH}"\nprint("Model directory:", model_path)\nprint("Circular TSV:", TSV_PATH)\nimport importlib.util\n\nrunner_path = Path("/kaggle/working/pcd_qwen7b/pcd_kaggle_runner.py")\nreference_path = Path("/kaggle/working/pcd_qwen7b/pcd_reference.py")\nspec = importlib.util.spec_from_file_location("pcd_kaggle_runner", runner_path)\nrunner = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(runner)\nCONFIG = {\n    "mode": MODE, "model_dir": str(model_path), "data_path": str(Path(TSV_PATH).resolve()),\n    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dataset_revision": DATASET_REVISION,\n    "output_dir": str(OUTPUT_DIR), "reference_path": str(reference_path),\n    "alpha": float(ALPHA), "num_workers": NUM_WORKERS, "seed": SEED,\n    "compute_baseline": COMPUTE_BASELINE, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS,\n    "sanity_qid": SANITY_QID, "resume_from": str(RESUME_FROM) if RESUME_FROM else None,\n}\nselected_rows = runner.prepare_run(CONFIG)\nprint("Selected rows:", len(selected_rows))\nfor rank in range(NUM_WORKERS):\n    plan = runner.read_json(OUTPUT_DIR / f"plan_gpu{rank}.json")\n    print(f"GPU {rank}: {len(plan)} rows")\nrunner.launch_workers(CONFIG)\nmetrics,records=runner.export_results(CONFIG)\nprint(metrics.to_string(index=False))\nif MODE==\'sanity\':\n    for row in records:\n        if row.get(\'diagnostic_camera_prediction\') is None or row.get(\'diagnostic_contrastive_prediction\') is None:\n            raise RuntimeError(\'Sanity failed: a branch has no valid option.\')\n        if row.get(\'reverse_direction_prediction\') is None:\n            raise RuntimeError(\'Sanity failed: reversed-direction output is invalid.\')\nprint(\'Done. Results and trace are ready for export.\')\n\nimport zipfile\narchive=OUTPUT_DIR.parent/(\'pcd_qwen7b_\'+MODE+\'_trace.zip\')\nwith zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as bundle:\n    for path in OUTPUT_DIR.rglob(\'*\'):\n        if path.is_file():bundle.write(path,path.relative_to(OUTPUT_DIR))\n    bundle.write(reference_path,\'pcd_reference.py\')\n    bundle.write(runner_path,\'pcd_kaggle_runner.py\')\nprint(\'Download:\',archive)\n'
values={k:globals()[k] for k in ['MODE','MODEL_ID','MODEL_REVISION','DATASET_ID','DATASET_REVISION','TSV_FILENAME','ALPHA','NUM_WORKERS','SEED','COMPUTE_BASELINE','MIN_PIXELS','MAX_PIXELS','MODEL_DIR','TSV_PATH','SANITY_QID','OUTPUT_DIR','RESUME_FROM']}
prefix='from pathlib import Path\n'+''.join(k+' = '+('Path('+repr(str(v))+')' if isinstance(v,Path) else repr(v))+'\n' for k,v in values.items())
job=WORK/'run_pcd.py'
job.write_text(prefix+driver,encoding='utf8')
subprocess.run([PY,'-u',str(job)],env=ENV,check=True)
